In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# after47 一次性分段反向定位
直接复用保存状态，不重建前缀、不改精度、不扫描参数。VAE 固定同一输入和 RGB 上游梯度反向两次；剩余 solver/Transformer 固定同一输入和第一份 VAE 上游梯度反向两次。诊断只用于定位数值差异，不产生或选择控制候选。
修复分段显存驻留：VAE 阶段把闲置 Transformer 移至 CPU；VAE 反向全部完成后，把 VAE 移至 CPU，再恢复 Transformer。仅在无待反向图的阶段边界迁移，保持模型精度、固定输入与原预算。split01 失败保留。


In [ ]:
from pathlib import Path
import sys, subprocess, json
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
SOURCE_BRANCH = 'dev/生成端公共关系载体/二维图像统计-持续生成约束'
SOURCE = Path('/content/public_statistic_phase2_source')
if SOURCE.exists(): raise FileExistsError('Use a fresh runtime; preserve existing source')
# 获取已发布开发分支，并记录实际使用的提交。
subprocess.run(['git','init',str(SOURCE)],check=True)
subprocess.run(['git','-C',str(SOURCE),'remote','add','origin',REPOSITORY_URL],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','--depth','1','origin',SOURCE_BRANCH],check=True)
subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','FETCH_HEAD'],check=True)
print('Source:',subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip())


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','diffusers','transformers','accelerate','ftfy','sentencepiece','safetensors','huggingface_hub','numpy','Pillow'],check=True)
subprocess.run(['ffmpeg','-version'],check=True)
# Keep Colab CUDA PyTorch. Actual versions are recorded by the worker.


## 固定预算与终点
12 次 Transformer、3 次 VAE、5 次求导调用（公共统计上游梯度1次、VAE2次、solver/Transformer2次）；另计最多240 block /26 chunk 重算。0 MP4，1800秒，零自动重试；无型号、版本或内存配额门禁。
本次之后根据证据定位具体错误、声明必要数值修正，或保留固定构造负结果并回到方法问题；不自动扩大数值诊断。当前冻结更新、run08负结果均不改变。CPU测试仅验证分段VJP与整链一致，真实GPU定位尚未执行。


In [ ]:
BASE=Path('/content/drive/MyDrive/Video-WM/public-statistic-phase2')
CONFIG=BASE/'run08/config.json'
REFERENCE=BASE/'after47-diagnostic01/after47_state_and_gradient.pt'
OUTPUT=BASE/'after47-split02'
for path in (CONFIG,REFERENCE):
    if not path.is_file(): raise FileNotFoundError(path)
if OUTPUT.exists(): raise FileExistsError(str(OUTPUT))
print('Source config:',CONFIG.read_text())
print('Budget: 12 transformer / 3 VAE / 5 backward; 240 block / 26 chunk replay; 0 MP4; 1800 seconds')


In [ ]:
import os, signal
command=[sys.executable,'-m','experiments.public_statistic.run_after47_split','--config',str(CONFIG),'--reference',str(REFERENCE),'--output',str(OUTPUT)]
process=subprocess.Popen(command,cwd=SOURCE,start_new_session=True)
try:
    returncode=process.wait()
except BaseException:
    # Cancel the launcher; it forwards cancellation and preserves worker results.
    try: process.send_signal(signal.SIGTERM)
    except ProcessLookupError: pass
    try: process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        try: os.killpg(process.pid,signal.SIGKILL)
        except ProcessLookupError: pass
        process.wait()
    raise
print('launcher exit',returncode)
print((OUTPUT/'result.json').read_text() if (OUTPUT/'result.json').exists() else 'No result file')
if returncode: raise subprocess.CalledProcessError(returncode,command)


## 回传
输出after47-split02。回传result.json、recomputation.json、execution_exit.json及execution.log；四个 .pt 文件分别保存两段的两次VJP。solver段报告在同一原始Δ上的g·Δ，VAE段比较完整梯度。没有新MP4，也没有候选接受判定。
